In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv(r"C:\Users\loneo\OneDrive\Documents\ev-charging-demand-segmentation\data\processed\ev_charging_cleaned.csv")

C:\Users\loneo\AppData\Local\Temp\ipykernel_26340\640505923.py:1: DtypeWarning: Columns (0: User ID, 1: County, 2: Model Number) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(r"C:\Users\loneo\OneDrive\Documents\ev-charging-demand-segmentation\data\processed\ev_charging_cleaned.csv")


In [3]:
df['Start Date'] = pd.to_datetime(df['Start Date'])
df['End Date'] = pd.to_datetime(df['End Date'])
df['Transaction Date (Pacific Time)'] = pd.to_datetime(df['Transaction Date (Pacific Time)'])

df['Total Duration (hh:mm:ss)'] = pd.to_timedelta(df['Total Duration (hh:mm:ss)'])

df['Charging Time (hh:mm:ss)'] = pd.to_timedelta(df['Charging Time (hh:mm:ss)'])

In [4]:
df['Start Date Year'] = pd.to_datetime(df['Start Date']).dt.year
df['Start Date Month'] = pd.to_datetime(df['Start Date']).dt.month
df['Start Date Date'] = pd.to_datetime(df['Start Date']).dt.date
df['Start Date Hour'] = pd.to_datetime(df['Start Date']).dt.hour
df['Start Date Dayofweek'] = pd.to_datetime(df['Start Date']).dt.dayofweek
df['Start Date is weekend'] = (pd.to_datetime(df['Start Date']).dt.dayofweek >= 5).astype(int)
df['Start Date quarter'] = pd.to_datetime(df['Start Date']).dt.quarter

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 259366 entries, 0 to 259365
Data columns (total 40 columns):
 #   Column                           Non-Null Count   Dtype          
---  ------                           --------------   -----          
 0   Station Name                     259366 non-null  str            
 1   MAC Address                      259366 non-null  str            
 2   Org Name                         259366 non-null  str            
 3   Start Date                       259366 non-null  datetime64[us] 
 4   Start Time Zone                  259366 non-null  str            
 5   End Date                         259366 non-null  datetime64[us] 
 6   End Time Zone                    259366 non-null  str            
 7   Transaction Date (Pacific Time)  259157 non-null  datetime64[us] 
 8   Total Duration (hh:mm:ss)        259366 non-null  timedelta64[us]
 9   Charging Time (hh:mm:ss)         259366 non-null  timedelta64[us]
 10  Energy (kWh)                     259366 non

In [6]:
station_day=df.groupby(['Station Name','Start Date Date']).agg({'Start Date':'count', 'Energy (kWh)':['sum','mean'],'Total Duration (hh:mm:ss)':'mean','Charging Time (hh:mm:ss)':'mean'})

In [7]:
station_day.columns = [
    'sessions_count',
    'total_energy',
    'avg_energy',
    'avg_total_duration',
    'avg_charging_time'
]

In [9]:
station_day.columns

Index(['sessions_count', 'total_energy', 'avg_energy', 'avg_total_duration',
       'avg_charging_time'],
      dtype='str')

In [10]:
station_day.reset_index(inplace=True)

In [11]:
station_day.head()


,Station Name,Start Date Date,sessions_count,total_energy,avg_energy,avg_total_duration,avg_charging_time
0,PALO ALTO CA / BRYANT # 1,2011-10-13,2,8.262052,4.131026,0 days 01:06:56.500000,0 days 01:06:48.500000
1,PALO ALTO CA / BRYANT # 1,2011-10-14,1,6.259466,6.259466,0 days 01:41:59,0 days 01:41:54
2,PALO ALTO CA / BRYANT # 1,2011-10-15,1,4.622894,4.622894,0 days 03:02:37,0 days 01:35:43
3,PALO ALTO CA / BRYANT # 1,2011-10-17,1,4.023361,4.023361,0 days 02:02:29,0 days 01:30:38
4,PALO ALTO CA / BRYANT # 1,2011-10-18,1,15.474944,15.474944,0 days 05:21:20,0 days 04:16:32


In [12]:
station_day.columns

Index(['Station Name', 'Start Date Date', 'sessions_count', 'total_energy',
       'avg_energy', 'avg_total_duration', 'avg_charging_time'],
      dtype='str')

In [13]:
print(station_day.shape)
station_day.info()

(55270, 7)
<class 'pandas.DataFrame'>
RangeIndex: 55270 entries, 0 to 55269
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   Station Name        55270 non-null  str            
 1   Start Date Date     55270 non-null  object         
 2   sessions_count      55270 non-null  int64          
 3   total_energy        55270 non-null  float64        
 4   avg_energy          55270 non-null  float64        
 5   avg_total_duration  55270 non-null  timedelta64[us]
 6   avg_charging_time   55270 non-null  timedelta64[us]
dtypes: float64(2), int64(1), object(1), str(1), timedelta64[us](2)
memory usage: 4.3+ MB


In [14]:
station_day['Start Date Date'] = pd.to_datetime(station_day['Start Date Date'])

station_day['avg_charging_time'] = (station_day['avg_charging_time'].dt.total_seconds() / 3600)

station_day['avg_total_duration'] = (station_day['avg_total_duration'].dt.total_seconds() / 3600)

In [15]:
station_day['sessions_count'].describe()

count    55270.000000
mean         4.692709
std          2.802635
min          1.000000
25%          2.000000
50%          4.000000
75%          6.000000
max         18.000000
Name: sessions_count, dtype: float64

In [16]:
station_day['sessions_count'].quantile(
    [0.90, 0.95, 0.97, 0.99]
)

0.90     9.0
0.95    10.0
0.97    11.0
0.99    12.0
Name: sessions_count, dtype: float64

In [17]:
station_day['Overload']=np.where(station_day['sessions_count']>= 11, 1, 0)

In [18]:
station_day['Overload'].value_counts()

Overload
0    53159
1     2111
Name: count, dtype: int64

In [19]:
station_day

,Station Name,Start Date Date,sessions_count,total_energy,avg_energy,avg_total_duration,avg_charging_time,Overload
0,PALO ALTO CA / BRYANT # 1,2011-10-13,2,8.262052,4.131026,1.115694,1.113472,0
1,PALO ALTO CA / BRYANT # 1,2011-10-14,1,6.259466,6.259466,1.699722,1.698333,0
2,PALO ALTO CA / BRYANT # 1,2011-10-15,1,4.622894,4.622894,3.043611,1.595278,0
3,PALO ALTO CA / BRYANT # 1,2011-10-17,1,4.023361,4.023361,2.041389,1.510556,0
4,PALO ALTO CA / BRYANT # 1,2011-10-18,1,15.474944,15.474944,5.355556,4.275556,0
...,...,...,...,...,...,...,...,...
55265,PALO ALTO CA / WEBSTER #3,2020-12-25,1,9.897000,9.897000,1.623611,1.620833,0
55266,PALO ALTO CA / WEBSTER #3,2020-12-26,1,10.359000,10.359000,1.711111,1.708889,0
55267,PALO ALTO CA / WEBSTER #3,2020-12-29,3,37.291000,12.430333,2.046204,2.035556,0
55268,PALO ALTO CA / WEBSTER #3,2020-12-30,2,47.706000,23.853000,3.972917,3.969167,0


In [20]:
station_day['Start Date Date']

0       2011-10-13
1       2011-10-14
2       2011-10-15
3       2011-10-17
4       2011-10-18
           ...    
55265   2020-12-25
55266   2020-12-26
55267   2020-12-29
55268   2020-12-30
55269   2020-12-31
Name: Start Date Date, Length: 55270, dtype: datetime64[s]

In [21]:
station_day['Start Date Month'] = station_day['Start Date Date'].dt.month
station_day['Start Date Dayofweek'] = station_day['Start Date Date'].dt.dayofweek
station_day['Start Date is weekend'] = (pd.to_datetime(station_day['Start Date Date']).dt.dayofweek >= 5).astype(int)
station_day['Start Date quarter'] = station_day['Start Date Date'].dt.quarter

In [22]:
station_day.sort_values(by=['Station Name','Start Date Date'],inplace=True)

In [23]:
station_day[['Station Name','Start Date Date']].head(20)

,Station Name,Start Date Date
0,PALO ALTO CA / BRYANT # 1,2011-10-13
1,PALO ALTO CA / BRYANT # 1,2011-10-14
2,PALO ALTO CA / BRYANT # 1,2011-10-15
3,PALO ALTO CA / BRYANT # 1,2011-10-17
4,PALO ALTO CA / BRYANT # 1,2011-10-18
5,PALO ALTO CA / BRYANT # 1,2011-10-19
6,PALO ALTO CA / BRYANT # 1,2011-10-20
7,PALO ALTO CA / BRYANT # 1,2011-10-21
8,PALO ALTO CA / BRYANT # 1,2011-10-24
9,PALO ALTO CA / BRYANT # 1,2011-10-25


In [24]:
station_day['lag_1_sessions']=station_day.groupby('Station Name')['sessions_count'].shift(1)
station_day['lag_7_sessions']=station_day.groupby('Station Name')['sessions_count'].shift(7)

In [27]:
station_day.isnull().sum()

Station Name               0
Start Date Date            0
sessions_count             0
total_energy               0
avg_energy                 0
avg_total_duration         0
avg_charging_time          0
Overload                   0
Start Date Month           0
Start Date Dayofweek       0
Start Date is weekend      0
Start Date quarter         0
lag_1_sessions            46
lag_7_sessions           266
dtype: int64

In [28]:
station_day['rolling_7_mean'] = station_day.groupby('Station Name')['sessions_count'].shift(1).transform(lambda x: x.rolling(7).mean())
station_day['rolling_7_std'] = station_day.groupby('Station Name')['sessions_count'].shift(1).transform(lambda x: x.rolling(7).std())


In [29]:
station_day.isnull().sum()

Station Name               0
Start Date Date            0
sessions_count             0
total_energy               0
avg_energy                 0
avg_total_duration         0
avg_charging_time          0
Overload                   0
Start Date Month           0
Start Date Dayofweek       0
Start Date is weekend      0
Start Date quarter         0
lag_1_sessions            46
lag_7_sessions           266
rolling_7_mean           266
rolling_7_std            266
dtype: int64

In [30]:
station_day['Overload'].value_counts()

Overload
0    53159
1     2111
Name: count, dtype: int64

In [31]:
station_day.dropna(inplace=True)


In [32]:
station_day['Overload'].value_counts()

Overload
0    52894
1     2110
Name: count, dtype: int64

In [33]:
print(station_day.shape)
station_day.sort_values(by=['Start Date Date'], inplace=True)

X=station_day.drop(columns=['Station Name','Start Date Date','sessions_count','Overload'])
y=station_day['Overload']

print(X.shape)
print(y.shape)

print(X.dtypes)

#Chronoplogical split

a=round(X.shape[0]*0.80)
X_train=X[0:a]
X_test=X[a:]
y_train=y[0:a]
y_test=y[a:]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(55004, 16)
(55004, 12)
(55004,)
total_energy             float64
avg_energy               float64
avg_total_duration       float64
avg_charging_time        float64
Start Date Month           int32
Start Date Dayofweek       int32
Start Date is weekend      int32
Start Date quarter         int32
lag_1_sessions           float64
lag_7_sessions           float64
rolling_7_mean           float64
rolling_7_std            float64
dtype: object
(44003, 12)
(44003,)
(11001, 12)
(11001,)


In [34]:
print(station_day[['Start Date Date']].head())
print(station_day[['Start Date Date']].tail())
print(station_day.iloc[a:]['Start Date Date'].min())
print(station_day.iloc[a:]['Start Date Date'].max())

      Start Date Date
16960      2011-08-05
16961      2011-08-06
16962      2011-08-07
16963      2011-08-08
20173      2011-08-09
      Start Date Date
48902      2020-12-31
25325      2020-12-31
26677      2020-12-31
12536      2020-12-31
55269      2020-12-31
2019-08-20 00:00:00
2020-12-31 00:00:00


In [35]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

lr_model = LogisticRegression(C=0.1, random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")

ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
ada_model.fit(X_train, y_train)
y_pred = ada_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")


rf_model = RandomForestClassifier(n_estimators=100, max_depth=5,class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")

xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=sum(y_train == 0) / sum(y_train == 1), random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")

C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-Score : 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10850
           1       1.00      1.00      1.00       151

    accuracy                           1.00     11001
   macro avg       1.00      1.00      1.00     11001
weighted avg       1.00      1.00      1.00     11001



Accuracy : 0.9862739750931734
Precision: 0.0
Recall   : 0.0
F1-Score : 0.0

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     10850
           1       0.00      0.00      0.00       151

    accuracy                           0.99     11001
   macro avg       0.49      0.50      0.50     11001
weighted avg       0.97      0.99      0.98     11001





C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\loneo\anaconda3\Lib\site-pa

Accuracy : 0.9707299336423961
Precision: 0.30612244897959184
Recall   : 0.8940397350993378
F1-Score : 0.4560810810810811

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     10850
           1       0.31      0.89      0.46       151

    accuracy                           0.97     11001
   macro avg       0.65      0.93      0.72     11001
weighted avg       0.99      0.97      0.98     11001



Accuracy : 0.9996363966912098
Precision: 0.9803921568627451
Recall   : 0.9933774834437086
F1-Score : 0.9868421052631579

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10850
           1       0.98      0.99      0.99       151

    accuracy                           1.00     11001
   macro avg       0.99      1.00      0.99     11001
weighted avg       1.00      1.00      1.00     11001





In [36]:
print(station_day.shape)
station_day.sort_values(by=['Start Date Date'], inplace=True)

X=station_day.drop(columns=['Station Name','Start Date Date','sessions_count','Overload','total_energy','avg_energy','avg_total_duration','avg_charging_time'])
y=station_day['Overload']

print(X.shape)
print(y.shape)

print(X.dtypes)

#Chronoplogical split

a=round(X.shape[0]*0.80)
X_train=X[0:a]
X_test=X[a:]
y_train=y[0:a]
y_test=y[a:]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(55004, 16)
(55004, 8)
(55004,)
Start Date Month           int32
Start Date Dayofweek       int32
Start Date is weekend      int32
Start Date quarter         int32
lag_1_sessions           float64
lag_7_sessions           float64
rolling_7_mean           float64
rolling_7_std            float64
dtype: object
(44003, 8)
(44003,)
(11001, 8)
(11001,)


In [37]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier




lr_model = LogisticRegression(C=0.1, random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
y_prob = lr_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")





ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
ada_model.fit(X_train, y_train)
y_pred = ada_model.predict(X_test)
y_prob = ada_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")




rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")





xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=sum(y_train == 0) / sum(y_train == 1), random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\n")

Accuracy : 0.9875465866739387
Precision: 0.5631067961165048
Recall   : 0.38666666666666666
F1-Score : 0.45849802371541504
ROC-AUC  : 0.9742063711485885

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     10851
           1       0.56      0.39      0.46       150

    accuracy                           0.99     11001
   macro avg       0.78      0.69      0.73     11001
weighted avg       0.99      0.99      0.99     11001



Accuracy : 0.9863648759203709
Precision: 0.0
Recall   : 0.0
F1-Score : 0.0
ROC-AUC  : 0.9727822320523454

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     10851
           1       0.00      0.00      0.00       150

    accuracy                           0.99     11001
   macro avg       0.49      0.50      0.50     11001
weighted avg       0.97      0.99      0.98     11001





C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\loneo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\loneo\anaconda3\Lib\site-pa

Accuracy : 0.9570039087355695
Precision: 0.21616871704745166
Recall   : 0.82
F1-Score : 0.34214186369958277
ROC-AUC  : 0.9755460326237212

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     10851
           1       0.22      0.82      0.34       150

    accuracy                           0.96     11001
   macro avg       0.61      0.89      0.66     11001
weighted avg       0.99      0.96      0.97     11001



Accuracy : 0.9644577765657667
Precision: 0.2515463917525773
Recall   : 0.8133333333333334
F1-Score : 0.384251968503937
ROC-AUC  : 0.9705326698000184

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     10851
           1       0.25      0.81      0.38       150

    accuracy                           0.96     11001
   macro avg       0.62      0.89      0.68     11001
weighted avg       0.99      0.96      0.97     11001





In [38]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance

,feature,importance
6,rolling_7_mean,0.346135
5,lag_7_sessions,0.334663
4,lag_1_sessions,0.210113
7,rolling_7_std,0.066429
1,Start Date Dayofweek,0.026738
2,Start Date is weekend,0.011231
0,Start Date Month,0.003701
3,Start Date quarter,0.000990


In [40]:
from sklearn.model_selection import TimeSeriesSplit

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print(station_day.shape)
station_day.sort_values(by=['Start Date Date'], inplace=True)

X=station_day.drop(columns=['Station Name','Start Date Date','sessions_count','Overload','total_energy','avg_energy','avg_total_duration','avg_charging_time'])
y=station_day['Overload']

print(X.shape)
print(y.shape)

tscv = TimeSeriesSplit(n_splits=5)
model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)

precisions = []
recalls = []
f1s = []
roc_aucs = []
accuracies = []

fold = 1

for train_index, test_index in tscv.split(X):
    X_train = X.iloc[train_index]
    X_test  = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test  = y.iloc[test_index]
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate ALL metrics, including accuracy
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    # Append ALL metrics to their lists
    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)
    roc_aucs.append(roc)
    
    print("Completed Fold", fold)
    fold = fold + 1

print("\nFINAL RESULTS ACROSS 5 FOLDS")

print("Average Accuracy  :", np.mean(accuracies))
print("Std Dev Accuracy  :", np.std(accuracies))

print("Average Precision :", np.mean(precisions))
print("Std Dev Precision :", np.std(precisions))

print("Average Recall    :", np.mean(recalls))
print("Std Dev Recall    :", np.std(recalls))

print("Average F1-Score  :", np.mean(f1s))
print("Std Dev F1-Score  :", np.std(f1s))

print("Average ROC-AUC   :", np.mean(roc_aucs))
print("Std Dev ROC-AUC   :", np.std(roc_aucs))

(55004, 16)
(55004, 8)
(55004,)
Completed Fold 1
Completed Fold 2
Completed Fold 3
Completed Fold 4
Completed Fold 5

FINAL RESULTS ACROSS 5 FOLDS
Average Accuracy  : 0.8946220137449548
Std Dev Accuracy  : 0.06274170360894554
Average Precision : 0.2447300077625978
Std Dev Precision : 0.054447752508573824
Average Recall    : 0.8652525865993825
Std Dev Recall    : 0.032837685720801806
Average F1-Score  : 0.3787707992233245
Std Dev F1-Score  : 0.06561048014575309
Average ROC-AUC   : 0.940910303544784
Std Dev ROC-AUC   : 0.03113043919853146


In [41]:
!pip install optuna

In [42]:
import numpy as np
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

print(station_day.shape)
station_day.sort_values(by=['Start Date Date'], inplace=True)

X=station_day.drop(columns=['Station Name','Start Date Date','sessions_count','Overload','total_energy','avg_energy','avg_total_duration','avg_charging_time'])
y=station_day['Overload']

print(X.shape)
print(y.shape)


def objective(trial):
    n_estimators = trial.suggest_categorical("n_estimators", [100, 200, 300, 500])
    max_depth = trial.suggest_categorical("max_depth", [3, 5, 7, 10, 15, 20])
    min_samples_split = trial.suggest_categorical("min_samples_split", [2, 5, 10, 20])
    min_samples_leaf = trial.suggest_categorical("min_samples_leaf", [1, 2, 4, 8])
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])
    
    model = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,
        max_features=max_features,class_weight='balanced',n_jobs=-1,random_state=42)
    
    tscv = TimeSeriesSplit(n_splits=5)
    f1_scores_for_trial = []
    
    for train_index, test_index in tscv.split(X):
        X_train = X.iloc[train_index]
        X_test  = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test  = y.iloc[test_index]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        f1 = f1_score(y_test, y_pred)
        f1_scores_for_trial.append(f1)
        
    return np.mean(f1_scores_for_trial)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("BEST PARAMETERS FOUND BY OPTUNA:")
for key, value in study.best_params.items():
    print(key, ":", value)



best_model = RandomForestClassifier(**study.best_params, class_weight='balanced', random_state=42)

tscv = TimeSeriesSplit(n_splits=5)

accuracies = []
precisions = []
recalls = []
f1s = []
roc_aucs = []

fold = 1

for train_index, test_index in tscv.split(X):
    X_train = X.iloc[train_index]
    X_test  = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test  = y.iloc[test_index]
    
    best_model.fit(X_train, y_train)
    
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)
    roc_aucs.append(roc)
    
    print("Completed Fold", fold)
    fold = fold + 1

print("\nFINAL RESULTS ACROSS 5 FOLDS (TUNED MODEL)")

print("Average Accuracy  :", np.mean(accuracies))
print("Std Dev Accuracy  :", np.std(accuracies))

print("Average Precision :", np.mean(precisions))
print("Std Dev Precision :", np.std(precisions))

print("Average Recall    :", np.mean(recalls))
print("Std Dev Recall    :", np.std(recalls))

print("Average F1-Score  :", np.mean(f1s))
print("Std Dev F1-Score  :", np.std(f1s))

print("Average ROC-AUC   :", np.mean(roc_aucs))
print("Std Dev ROC-AUC   :", np.std(roc_aucs))


[I 2026-06-23 21:32:13,657] A new study created in memory with name: no-name-45133662-530d-4648-a66e-14ade173c072


(55004, 16)
(55004, 8)
(55004,)


[I 2026-06-23 21:32:45,823] Trial 0 finished with value: 0.4098366514722297 and parameters: {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 20, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.4098366514722297.
[I 2026-06-23 21:33:18,470] Trial 1 finished with value: 0.40220310174282325 and parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.4098366514722297.
[I 2026-06-23 21:33:25,505] Trial 2 finished with value: 0.3872298902677152 and parameters: {'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.4098366514722297.
[I 2026-06-23 21:34:10,302] Trial 3 finished with value: 0.383453573961552 and parameters: {'n_estimators': 500, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.4098366514722

BEST PARAMETERS FOUND BY OPTUNA:
n_estimators : 300
max_depth : 20
min_samples_split : 10
min_samples_leaf : 8
max_features : log2
Completed Fold 1
Completed Fold 2
Completed Fold 3
Completed Fold 4
Completed Fold 5

FINAL RESULTS ACROSS 5 FOLDS (TUNED MODEL)
Average Accuracy  : 0.934416930293444
Std Dev Accuracy  : 0.04766824281455875
Average Precision : 0.34694427487495433
Std Dev Precision : 0.038790911244452254
Average Recall    : 0.6503667928042368
Std Dev Recall    : 0.18114107725483658
Average F1-Score  : 0.43791024196871386
Std Dev F1-Score  : 0.0647879813463336
Average ROC-AUC   : 0.9376151680479448
Std Dev ROC-AUC   : 0.02943714944779324


In [80]:
import os
import joblib
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=8,
    max_features='log2',
    class_weight='balanced',
    n_jobs=-1,  
    random_state=42
)

tscv = TimeSeriesSplit(n_splits=5)

accuracies = []
precisions = []
recalls = []
f1s = []
roc_aucs = []

fold = 1



for train_index, test_index in tscv.split(X):
    X_train = X.iloc[train_index]
    X_test  = X.iloc[test_index]
    y_train = y.iloc[train_index]
    y_test  = y.iloc[test_index]
    
    rf_final.fit(X_train, y_train)
    
    y_pred = rf_final.predict(X_test)
    y_prob = rf_final.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)
    roc_aucs.append(roc)
    
    print("Completed Fold", fold)
    fold = fold + 1

print("\nFINAL EVALUATION RESULTS")
print("Average Accuracy  :", np.mean(accuracies))
print("Average Precision :", np.mean(precisions))
print("Average Recall    :", np.mean(recalls))
print("Average F1-Score  :", np.mean(f1s))
print("Average ROC-AUC   :", np.mean(roc_aucs))


save_dir = r"C:\Users\loneo\OneDrive\Documents\ev-charging-demand-segmentation\model"
file_name = "rf_tuned_model.joblib"
full_path = os.path.join(save_dir, file_name)

os.makedirs(save_dir, exist_ok=True)
joblib.dump(rf_final, full_path)

print("Model trained on final fold and successfully saved to:")
print(full_path)

Completed Fold 1
Completed Fold 2
Completed Fold 3
Completed Fold 4
Completed Fold 5

FINAL EVALUATION RESULTS
Average Accuracy  : 0.934416930293444
Average Precision : 0.34694427487495433
Average Recall    : 0.6503667928042368
Average F1-Score  : 0.43791024196871386
Average ROC-AUC   : 0.9376151680479448
Model trained on final fold and successfully saved to:
C:\Users\loneo\OneDrive\Documents\ev-charging-demand-segmentation\model\rf_tuned_model.joblib


In [82]:
import os

save_dir = r"C:\Users\loneo\OneDrive\Documents\ev-charging-demand-segmentation\optuna_results"
file_name = "rf_optuna_metrics.txt"
full_path = os.path.join(save_dir, file_name)

os.makedirs(save_dir, exist_ok=True)

results_text = """Best Random Forest Parameters

n_estimators = 300
max_depth = 20
min_samples_split = 10
min_samples_leaf = 8
max_features = log2

Cross-Validation Results (TimeSeriesSplit)

Accuracy  : 0.9344 ± 0.0477
Precision : 0.3469 ± 0.0388
Recall    : 0.6504 ± 0.1811
F1-Score  : 0.4379 ± 0.0648
ROC-AUC   : 0.9376 ± 0.0294"""

with open(full_path, "w", encoding="utf-8") as file:
    file.write(results_text)

print("Results successfully saved to:")
print(full_path)

Results successfully saved to:
C:\Users\loneo\OneDrive\Documents\ev-charging-demand-segmentation\optuna_results\rf_optuna_metrics.txt
